In [5]:
!pip install moya-ai

In [6]:
!pip install python-docx
!pip install PyPDF2


In [7]:
import os
from dataclasses import dataclass
from typing import Dict, Any, Optional
import docx
import PyPDF2

# Moya imports
from moya.agents.openai_agent import OpenAIAgent, OpenAIAgentConfig
from moya.classifiers.llm_classifier import LLMClassifier
from moya.memory.in_memory_repository import InMemoryRepository
from moya.orchestrators.multi_agent_orchestrator import MultiAgentOrchestrator
from moya.registry.agent_registry import AgentRegistry
from moya.conversation.thread import Thread
from moya.conversation.message import Message
from moya.tools.tool_registry import ToolRegistry


# === File Readers ===
def read_docx(path):
    try:
        doc = docx.Document(path)
        return "\n".join(p.text for p in doc.paragraphs if p.text.strip())
    except Exception as e:
        return f"⚠️ Failed to read DOCX: {e}"


def read_pdf(path):
    try:
        with open(path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            return "\n".join(p.extract_text() for p in reader.pages if p.extract_text())
    except Exception as e:
        return f"⚠️ Failed to read PDF: {e}"

# --- Unified MemoryTool ---
@dataclass
class MemoryTool:
    name: str = "memory_tool"
    description: str = "Tool for memory operations: store and retrieve messages."
    repo: InMemoryRepository = InMemoryRepository()

    def __call__(self, **kwargs) -> Any:
        method_name = kwargs.pop("method_name", None)
        if method_name is None:
            raise Exception("method_name parameter is required")

        thread_id = kwargs.pop("thread_id", None)
        if thread_id is None:
            raise Exception("thread_id parameter is required")

        if method_name == "store_message":
            sender = kwargs.pop("sender", None)
            content = kwargs.pop("content", None)
            metadata = kwargs.pop("metadata", None)
            if sender is None or content is None:
                raise Exception("sender and content are required for store_message")
            return self.store_message(thread_id=thread_id, sender=sender, content=content, metadata=metadata)

        elif method_name == "get_last_n_messages":
            n = kwargs.pop("n", 5)
            return self.get_last_n_messages(thread_id=thread_id, n=n)

        elif method_name == "get_thread_summary":
            return self.get_thread_summary(thread_id=thread_id)

        else:
            raise Exception(f"Unknown method: {method_name}")

    @property
    def function(self):
        return self.__call__

    @property
    def parameters(self):
        return {
            "method_name": {
                "type": "string",
                "description": "The memory method to call: store_message, get_last_n_messages, or get_thread_summary"
            },
            "thread_id": {
                "type": "string",
                "description": "Thread ID for the conversation"
            },
            "sender": {
                "type": "string",
                "description": "Sender of the message (for store_message)",
                "nullable": True
            },
            "content": {
                "type": "string",
                "description": "Message content (for store_message)",
                "nullable": True
            },
            "n": {
                "type": "integer",
                "description": "Number of recent messages to fetch (for get_last_n_messages)",
                "nullable": True
            }
        }

    @property
    def required(self):
        return ["method_name", "thread_id"]

    def store_message(self, thread_id: str, sender: str, content: str, metadata: Optional[Dict[str, Any]] = None) -> str:
        thread = self.repo.get_thread(thread_id) or Thread(thread_id=thread_id)
        if not self.repo.get_thread(thread_id):
            self.repo.create_thread(thread)
        message = Message(thread_id=thread_id, sender=sender, content=content, metadata=metadata or {})
        self.repo.append_message(thread_id, message)
        return "Message stored."

    def get_last_n_messages(self, thread_id: str, n: int = 5) -> str:
        thread = self.repo.get_thread(thread_id)
        if not thread:
            return "[]"
        return str([m.to_dict() for m in thread.get_last_n_messages(n)])

    def get_thread_summary(self, thread_id: str) -> str:
        thread = self.repo.get_thread(thread_id)
        if not thread:
            return "No conversation found."
        return "\n".join(f"{m.sender}: {m.content}" for m in thread.messages[-10:])


# --- Tool Registry Setup ---
def setup_tool_registry() -> ToolRegistry:
    tool_registry = ToolRegistry()
    memory_tool = MemoryTool()
    tool_registry.register_tool(memory_tool)
    return tool_registry




# --- Agent Creation ---
def resume_agent(tool_registry):
    """Create a resume-building OpenAI agent with ATS scoring and resume generation capabilities."""
    config = OpenAIAgentConfig(
        agent_name="resume_agent",
        description="An agent that evaluates resumes, scores them for ATS compatibility, and generates optimized resumes based on job descriptions.",
        api_key=os.getenv("OPENAI_API_KEY"),
        model_name="gpt-4o-2024-11-20",
        agent_type="ChatAgent",
        tool_registry=tool_registry,
        is_streaming=True,
        system_prompt=(
            "You are a professional resume assistant specialized in resume optimization for job applications.\n\n"
            "Your tasks include:\n"
            "- Analyzing the user's resume and job description to compute an ATS (Applicant Tracking System) score out of 100.\n"
            "- Providing clear, actionable suggestions to improve the resume (e.g., adding missing keywords, aligning skills, rephrasing bullet points).\n"
            "- Be structured and concise in your feedback.\n"
            "- Emphasize keywords and measurable impact\n"
            "- Use professional language throughout.\n"
            "- Only assist with resume and job application tasks unless explicitly told otherwise.\n"
        ),
    )
    return OpenAIAgent(config)


def interview_trainer_agent(tool_registry):
    """Create an interview training agent that simulates interviews and gives feedback."""
    config = OpenAIAgentConfig(
        agent_name="interview_trainer_agent",
        description="An agent that conducts mock interviews and provides feedback on responses.",
        api_key=os.getenv("OPENAI_API_KEY"),
        model_name="gpt-4o-2024-11-20",
        agent_type="ChatAgent",
        tool_registry=tool_registry,
        is_streaming=True,
        
        system_prompt=("You are a professional interview training agent simulating realistic mock interviews based on the candidate’s resume and job description.\n\n"
"- Ask the user what type of interview questions they’d like to practice (e.g., HR, behavioral, technical, mixed).\n"
"- Then ask 5–10 tailored questions of that type, one at a time.\n"
"- After each answer, give structured feedback:\n"
"   • Strength\n"
"   • Weakness\n"
"   • Improvement tip\n"
"- After all questions, give a performance summary:\n"
"   • Strengths\n"
"   • Areas to improve\n"
"   • Score out of 10\n"
"- Be supportive, constructive, and professional.\n"
"Start by greeting the user, asking if they’re ready, and what question type they want to focus on."
        ),
    )
    return OpenAIAgent(config)


def classifier_agent(tool_registry):
    """Create a classifier agent that routes to resume or interview agents based on the user's message."""
    config = OpenAIAgentConfig(
        agent_name="classifier_agent",
        description="Classifies user messages to the resume agent or interview agent based on task intent.",
        api_key=os.getenv("OPENAI_API_KEY"),
        model_name="gpt-4o-2024-11-20",
        agent_type="ChatAgent",
        tool_registry=tool_registry,
        is_streaming=True,
        system_prompt=(
            "You are a classifier agent.\n\n"
            "Your task is to analyze the user's input and determine which agent should handle the request.\n\n"
            "Respond ONLY with one of the following agent names:\n"
            "- 'resume_agent': If the user asks about building,improving, reviewing, editing, scoring, or optimizing a resume/CV, "
            "or mentions job description alignment.\n"
            "- 'interview_trainer_agent': If the user asks for mock interviews, interview questions, practice responses, "
            "or interview feedback.\n"
            "- 'none': If the request is unrelated to resume or interview tasks.\n\n"
            "Examples:\n"
            "- \"Can you help me improve my resume for a data analyst job?\" → resume_agent\n"
            "- \"I want to practice interview questions for a frontend role\" → interview_trainer_agent\n"
            "- \"What’s the weather like today?\" → none\n"
        )
    )
    return OpenAIAgent(config)


# --- Orchestrator Setup ---
def setup_orchestrator():
    tool_registry = setup_tool_registry()
    registry = AgentRegistry()

    # Register the main agents
    registry.register_agent(resume_agent(tool_registry))
    registry.register_agent(interview_trainer_agent(tool_registry))

    # Set up classifier
    classifier = classifier_agent(tool_registry)

    return MultiAgentOrchestrator(
        agent_registry=registry,
        classifier=LLMClassifier(classifier, default_agent="resume_agent"),
        default_agent_name="resume_agent"
    )

# --- Main Chat Loop ---
def main():
    orchestrator = setup_orchestrator()
    thread_id = "resume_demo"

    print("👋 Welcome to Your AI Career Coach!\n")
    print("Ready to land your dream job?")
    print("We're here to build your perfect resume and help you ace interviews with confidence.\n")

    print("📝 Tailored Resume Builder")
    print("Just tell us about your experience and the job you’re aiming for—we’ll craft a polished, customized resume in minutes.\n")

    print("🎤 Mock Interview Trainer")
    print("Practice with realistic interview questions, get real-time feedback, and track your progress with smart memory insights.\n")

    print("Let’s get started—your career breakthrough begins here!\n")
    print("Upload resume and job description as:")
    print("/resume path/to/resume.docx or .pdf")
    print("/jd path/to/job_description.docx or .pdf")
    print("Type 'exit' to quit.")

    resume_text, job_text = "", ""

    def generate_prompt(resume_text, job_text, summary, user_input):
        full_prompt = (
            f"Resume:\n{resume_text}\n\n"
            f"Job Description:\n{job_text}\n\n"
            f"Conversation so far:\n{summary}\n\n"
            f"User: {user_input}"
        )
        return full_prompt

    while True:
        user_input = input("\nYou: ").strip()

        if user_input.lower() == "exit":
            print("👋 Goodbye!")
            break

        if user_input.startswith("/resume "):
            path = user_input[len("/resume "):].strip()
            resume_text = read_docx(path) if path.endswith(".docx") else read_pdf(path)
            print("✅ Resume uploaded.")
            continue

        if user_input.startswith("/jd "):
            path = user_input[len("/jd "):].strip()
            job_text = read_docx(path) if path.endswith(".docx") else read_pdf(path)
            print("✅ Job description uploaded.")
            continue

        if not resume_text or not job_text:
            print("❌ Please upload both resume and job description before proceeding.")
            continue

        collected = []
        agent = orchestrator.agent_registry.get_agent("resume_agent")

        # Memory storage
        try:
            agent.call_tool("memory_tool", method_name="store_message",
                            thread_id=thread_id, sender="user", content=user_input)
        except Exception as e:
            print(f"[Memory Error]: {e}")

        # Memory retrieval
        try:
            context = agent.call_tool("memory_tool", method_name="get_last_n_messages",
                                      thread_id=thread_id, n=5)
            summary = agent.call_tool("memory_tool", method_name="get_thread_summary",
                                      thread_id=thread_id)
        except Exception as e:
            print(f"[Context Error]: {e}")
            context, summary = "", ""

        # Compose prompt
        full_prompt = generate_prompt(resume_text, job_text, summary, user_input)

        # Generate assistant response
        print("Assistant: ", end="", flush=True)
        orchestrator.orchestrate(
            thread_id=thread_id,
            user_message=full_prompt,
            stream_callback=lambda x: print(x, end="", flush=True)
        )
        print()

        # Show context info
        print("\n--- Last 5 messages ---")
        print(context)
        print("\n--- Conversation Summary ---")
        print(summary)
        print("------------------------\n")


if __name__ == "__main__":
    os.environ["OPENAI_API_KEY"] = "SECRET KEY"
    main()


👋 Welcome to Your AI Career Coach!

Ready to land your dream job?
We're here to build your perfect resume and help you ace interviews with confidence.

📝 Tailored Resume Builder
Just tell us about your experience and the job you’re aiming for—we’ll craft a polished, customized resume in minutes.

🎤 Mock Interview Trainer
Practice with realistic interview questions, get real-time feedback, and track your progress with smart memory insights.

Let’s get started—your career breakthrough begins here!

Upload resume and job description as:
/resume path/to/resume.docx or .pdf
/jd path/to/job_description.docx or .pdf
Type 'exit' to quit.



You:  /resume /kaggle/input/sample-resumes/Rajesh_Kumar_Weak_ATS_Resume.docx


✅ Resume uploaded.



You:  /jd /kaggle/input/sample-jds/Data Analyst  Entry Level.pdf


✅ Job description uploaded.


...


You:  "A JOIN in SQL is used when we want to combine all the data from different databases. It mixes up all the rows randomly from two tables without needing a condition. For example, if you use JOIN, it just adds all the data from both tables side by side, like copy-paste.


Assistant: [interview_trainer_agent] Great! Thank you for sharing. It looks like you're ready to dive into technical questions. Let’s delve into common Data Analyst technical questions tailored to your background and the job posting.

Let’s start with the first technical question:

### Question 1:
**Can you explain the difference between a Primary Key and a Foreign Key in the context of SQL databases?**

You’ve already shared your response to this, so let me evaluate:

**Your Answer:**  
*"Primary keys and foreign keys are the same thing. They are both used to make sure that tables in a database are connected. A primary key is any column in a table, and a foreign key is just another term for the same thing but in a different table. You can have as many primary keys as you want in one table, and foreign keys don't really need to match anything. They just help store extra information."*

---

### Feedback:
1. **Strength:**
   - It's good that you recognize that primary keys and foreign k


You:  "A JOIN in SQL is used when we want to combine all the data from different databases. It mixes up all the rows randomly from two tables without needing a condition. For example, if you use JOIN, it just adds all the data from both tables side by side, like copy-paste.


Assistant: [resume_agent] ### Current ATS Score: **65/100**
The resume moderately aligns with the job description for the **Data Analyst - Entry Level** position at Fractal Analytics but needs significant enhancements for a stronger alignment.

---

### Suggestions for Optimization

#### 1. **Objective Statement**
   - **Current Issue:** Vague, lacks customization and keywords.
   - **Suggestion:** Replace with:  
     *"Aspiring Data Analyst with a strong foundation in Mathematics and hands-on experience in Python, SQL, and Excel. Seeking to leverage data cleaning, visualization, and analytical skills to drive actionable insights at Fractal Analytics."*

#### 2. **Education**
   - No major changes required. Highlight your CGPA clearly as done. However, consider bulleted format for readability:
     - **B.Sc. in Mathematics**, XYZ College, Delhi University (2023)  
       - CGPA: 7.2

#### 3. **Skills**
   - **Missing Keywords from Job Description:** Power BI, Tableau, Data Visualizati

KeyboardInterrupt: Interrupted by user

In [3]:
from docx import Document

# Resume 1 - Data Analyst (Weak ATS)
resume1 = Document()
resume1.add_heading("Rajesh Kumar", level=1)
resume1.add_paragraph("Email: rajesh123@gmail.com | Phone: +91-9876543210")
resume1.add_paragraph("LinkedIn: linkedin.com/in/rajeshkumar")
resume1.add_heading("Objective", level=2)
resume1.add_paragraph("Looking for a job in a good company where I can use my skills and grow. I am interested in data and computers.")
resume1.add_heading("Education", level=2)
resume1.add_paragraph("B.Sc. in Mathematics\n XYZ College, Delhi University\nGraduated: 2023\nCGPA: 7.2")
resume1.add_heading("Skills", level=2)
resume1.add_paragraph("MS Excel, Python, SQL, Team work, Communication, Computer basics")
resume1.add_heading("Projects", level=2)
resume1.add_paragraph("1. Data Cleaning\n- Cleaned a dataset in Excel.\n- Removed missing values and duplicate rows.")
resume1.add_paragraph("2. Online Sales Data\n- Worked with e-commerce data.\n- Made charts using Excel.")
resume1.add_heading("Certifications", level=2)
resume1.add_paragraph("Python for Beginners – Coursera\nExcel Basics – Udemy")
resume1.add_heading("Experience", level=2)
resume1.add_paragraph("Intern – Computer Lab Assistant\nABC Institute, Jan 2023 – Apr 2023\n- Helped with system maintenance\n- Assisted in lab inventory\n- Supported students during lab hours")
resume1.add_heading("Achievements", level=2)
resume1.add_paragraph("Participated in college quiz competition\nVolunteered in annual fest")
resume1.add_heading("Languages", level=2)
resume1.add_paragraph("English, Hindi")
resume1.add_heading("Hobbies", level=2)
resume1.add_paragraph("Watching tutorials, Playing chess")

# Resume 2 - Software Engineer (Weak ATS)
resume2 = Document()
resume2.add_heading("Sneha Sharma", level=1)
resume2.add_paragraph("Email: sneha.sharma@email.com | Phone: +91-9988776655")
resume2.add_paragraph("GitHub: github.com/snehasharma")
resume2.add_heading("Career Objective", level=2)
resume2.add_paragraph("To become a good software engineer in a reputed company. I want to work on exciting projects and learn.")
resume2.add_heading("Education", level=2)
resume2.add_paragraph("Bachelor of Technology – Computer Science\nABC Engineering College, Hyderabad\nGraduated: 2024\nCGPA: 6.9")
resume2.add_heading("Skills", level=2)
resume2.add_paragraph("Java, HTML, Database, Basic C, MS Word, Programming Logic")
resume2.add_heading("Projects", level=2)
resume2.add_paragraph("1. Student Record Management\n- Made using Java.\n- Stores student data.")
resume2.add_paragraph("2. Personal Website\n- Built a basic portfolio using HTML.\n- Deployed on free hosting.")
resume2.add_heading("Certifications", level=2)
resume2.add_paragraph("Java Programming – Great Learning\nWeb Development Basics – YouTube Course")
resume2.add_heading("Internship", level=2)
resume2.add_paragraph("IT Intern – Softsys Pvt Ltd\nMay 2023 – July 2023\n- Shadowed developers\n- Did basic documentation\n- Tested login page manually")
resume2.add_heading("Soft Skills", level=2)
resume2.add_paragraph("Hardworking, Willing to learn, Honest")
resume2.add_heading("Personal Info", level=2)
resume2.add_paragraph("Date of Birth: 12 Oct 2002\nFather’s Name: Mr. R. Sharma\nLanguages: English, Hindi\nAddress: 56, ABC Nagar, Hyderabad")
resume2.add_heading("Declaration", level=2)
resume2.add_paragraph("I hereby declare that all the information given is true to the best of my knowledge.")

# Save both documents
path1 = "/kaggle/working/Rajesh_Kumar_Weak_ATS_Resume.docx"
path2 = "/kaggle/working/Sneha_Sharma_Weak_ATS_Resume.docx"
resume1.save(path1)
resume2.save(path2)

path1, path2


('/kaggle/working/Rajesh_Kumar_Weak_ATS_Resume.docx',
 '/kaggle/working/Sneha_Sharma_Weak_ATS_Resume.docx')